# Ambiente de Testes SQL com DuckDB

Este notebook é uma plataforma de laboratório para testar consultas SQL com DuckDB de forma ampla e profissional.

- Carrega automaticamente todos os arquivos CSV na pasta `data/`.
- Registra cada arquivo como uma tabela DuckDB para criar um ambiente relacional completo.
- Permite criar e validar consultas de seleção, agregação, joins, funções de janela e análises exploratórias.

> Além disso, existe uma pasta `sql_commands/` no repositório para que você salve seus scripts e exemplos SQL em arquivos `.sql` ou `.txt`. Isso torna seu material de aula mais organizado e reutilizável.


## 1. Preparar o ambiente

Execute esta célula para carregar as bibliotecas necessárias e criar uma função de apoio para executar consultas SQL no DuckDB.


Pressione Ctrl + Shift + P e digite "Jupyter: Select Kernel"
Escolha "Python Local"
Clique em "Restart Kernel" ou recarregue o arquivo

Observação:
Se houver múltiplas versões do Python, use o Python Local que funciona corretamente no seu ambiente. O VS Code pode sugerir outras versões (ex.: Python 3.13.5), mas nem sempre elas possuem os kernels registrados.

In [1]:
import importlib.util
import subprocess
import sys

#for pkg in ('duckdb', 'pandas'):
#    if importlib.util.find_spec(pkg) is None:
#        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])



for pkg in ('duckdb', 'pandas', 'ipykernel'):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])


import duckdb
import pandas as pd
from pathlib import Path

# Diretório de dados local
data_dir = Path('../data')
if not data_dir.exists():
    raise FileNotFoundError(f'Diretório não encontrado: {data_dir}')

csv_files = sorted(data_dir.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'Nenhum arquivo CSV encontrado em: {data_dir}')

# Conexão local em memória
con = duckdb.connect(database=':memory:')
loaded_tables = []
for csv_path in csv_files:
    table_name = csv_path.stem.lower()
    df = pd.read_csv(csv_path)
    con.register(table_name, df)
    loaded_tables.append(table_name)

last_result = None

def sql(query: str):
    global last_result
    last_result = con.execute(query).df()
    display(last_result)

## 2. Carregar os dados locais

Este notebook carrega todos os arquivos CSV presentes em `data/`.
Cada arquivo é registrado automaticamente no DuckDB como uma tabela cujo nome é o nome do arquivo (sem extensão).

Dessa forma, você pode testar `JOINs` e outras consultas SQL entre tabelas diferentes.

Por exemplo:

- `cliente.csv` → tabela `cliente`
- `produto.csv` → tabela `produto`
- `vendedor.csv` → tabela `vendedor`
- `vendas.csv` → tabela `vendas`

In [2]:
# Exibe as tabelas carregadas e um exemplo de dados - {loaded_tables[n]}
print(f'Arquivos CSV carregados como tabelas: {", ".join(loaded_tables)}')
print(f'Tabela de exemplo: {loaded_tables[1]}')
display(con.execute(f"SELECT * FROM {loaded_tables[1]} LIMIT 5").df())


Arquivos CSV carregados como tabelas: baby_names, menu_items, order_details, restaurant_db_data_dictionary, student_grades, students
Tabela de exemplo: menu_items


,menu_item_id,item_name,category,price
0,101,Hamburger,American,12.95
1,102,Cheeseburger,American,13.95
2,103,Hot Dog,American,9.00
3,104,Veggie Burger,American,10.50
4,105,Mac & Cheese,American,7.00


### Duplique este notebook para começar outro exercício e insira suas consultas SQL abaixo: (divirta-se :-) .

### 💡 Modelos de Referência (Snippets)

Utilize os padrões abaixo para construir suas análises:

**1. Filtro e Ordenação:**
```sql
sql("""
SELECT Nome, Total
FROM baby_names
WHERE Sexo = 'Menina'
ORDER BY Total DESC
""" )
```

**2. Agregação e Sumarização:**
```sql
sql("""
SELECT Sexo, COUNT(*) as Qtd_Nomes, SUM(Total) as Total_Bebes
FROM baby_names
GROUP BY Sexo
""" )
```

---

 Proj : Nome do Projeto Restaurant Orders

In [3]:
# Exemplo de consulta SQL itens Menu
sql(' SELECT * FROM menu_items LIMIT 5 ')

,menu_item_id,item_name,category,price
0,101,Hamburger,American,12.95
1,102,Cheeseburger,American,13.95
2,103,Hot Dog,American,9.00
3,104,Veggie Burger,American,10.50
4,105,Mac & Cheese,American,7.00


#### How many categories have a maximum price below $15

#### What´s the max price for each category ?

In [6]:
sql(''' 
SELECT category, MAX(price) AS max_price
FROM menu_items
GROUP BY category
    ''')

,category,max_price
0,Asian,17.95
1,American,13.95
2,Mexican,14.95
3,Italian,19.95


#### How many max prices are less thas $15?

In [12]:
## subquery 
sql('''
   SELECT COUNT(*)
    FROM 
   
(SELECT category, MAX(price) AS max_price
FROM menu_items
GROUP BY category) AS mp -- salvando a subquery como mp
    
    
WHERE max_price < 15;
    ''')

,count_star()
0,2


In [17]:
# Mais simples, sem subquery
sql('''

SELECT category, MAX(price) AS max_price
FROM menu_items
GROUP BY category
HAVING MAX(price) < 15;

''')

,category,max_price
0,American,13.95
1,Mexican,14.95


In [21]:
# CTE

In [23]:
### 2. CTE: Multiple references 
sql('''

    -- Cria uma tabela temporária nomeada (mp)
WITH mp AS (SELECT category, MAX(price) AS max_price
            FROM menu_items
            GROUP BY category)
    
    -- Query principal
SELECT COUNT(*)
FROM mp
WHERE max_price < 15;
    
''')

,count_star()
0,2


In [24]:
## Reutilização (🔥 aqui entra o “Multiple references”)

sql('''

WITH mp AS (
    SELECT category, MAX(price) AS max_price
    FROM menu_items
    GROUP BY category
)

SELECT 
    COUNT(*) AS total_categorias,
    AVG(max_price) AS media_preco,
    MAX(max_price) AS maior_preco
FROM mp;
    
''')

,total_categorias,media_preco,maior_preco
0,4,16.7,19.95


### CTE : Multiple references

In [26]:
sql('''

    -- Cria uma tabela temporária nomeada (mp)
WITH mp AS (SELECT category, MAX(price) AS max_price
            FROM menu_items
            GROUP BY category)
    
    -- Query principal
SELECT COUNT(*)
FROM mp
WHERE max_price < (SELECT AVG(max_price) FROM mp);
    
''')

,count_star()
0,2


In [29]:
# CTE: Multipas tabelas
sql('''

    -- CTE : Multiplas tabelas (mp e ci)
WITH mp AS (SELECT category, MAX(price) AS max_price
            FROM menu_items
            GROUP BY category),
     ci AS (SELECT * 
            FROM menu_items
            WHERE item_name LIKE '%Chicken%') 
    
SELECT *
    FROM ci LEFT JOIN mp ON ci.category = mp.category;

''')

,menu_item_id,item_name,category,price,category_1,max_price
0,107,Orange Chicken,Asian,16.50,Asian,17.95
1,119,Chicken Torta,Mexican,11.95,Mexican,14.95
2,131,Chicken Parmesan,Italian,17.95,Italian,19.95
3,117,Chicken Burrito,Mexican,12.95,Mexican,14.95
4,115,Chicken Tacos,Mexican,11.95,Mexican,14.95
